### Part 1: Theory & Derivations (The Written Exam)

Write this exact computational graph and step-by-step derivation on your reference sheet.

**1. The Forward Pass (Classification)**

*   **Hidden Layer:** $Z_1 = X W_1 + b_1$
*   **Activation (Sigmoid):** $A_1 = \sigma(Z_1) = \frac{1}{1 + e^{-Z_1}}$
*   **Output Layer (Logits):** $Z_2 = A_1 W_2 + b_2$
*   **Softmax Probabilities:** $P_i = \frac{e^{Z_{2, i}}}{\sum_{j} e^{Z_{2, j}}}$ *(Converts raw logits to a probability distribution summing to 1)*
*   **Cross-Entropy Loss:** $L = -\frac{1}{N} \sum_{i=1}^{N} \sum_{k=1}^{C} T_{i,k} \log(P_{i,k})$ *(Where $T$ is the one-hot target matrix)*

**2. The Backward Pass (Derivations & Tensor Shapes)**
Assume $X \in \mathbb{R}^{N \times 2}$, $W_1 \in \mathbb{R}^{2 \times 5}$, $W_2 \in \mathbb{R}^{5 \times 3}$.

**Step 1: The Output Gradient (The Crucial Simplification)**
The derivative of Cross-Entropy loss with respect to the pre-Softmax logits ($Z_2$) simplifies beautifully to the predicted probabilities minus the true labels.

$$\frac{\partial L}{\partial Z_2} = \frac{1}{N}(P - T)$$
*(Shape Check: N × 3)*

**Step 2: Output Weights and Biases**

$$\frac{\partial L}{\partial W_2} = A_1^T \frac{\partial L}{\partial Z_2}$$
*(Shape Check: (5 × N) @ (N × 3) = 5 × 3. Matches $W_2$)*

$$\frac{\partial L}{\partial b_2} = \sum_{i=1}^{N} \left(\frac{\partial L}{\partial Z_2}\right)_i$$
*(Shape Check: 1 × 3)*

**Step 3: Hidden Error Propagation**

$$\frac{\partial L}{\partial A_1} = \frac{\partial L}{\partial Z_2} W_2^T$$
*(Shape Check: (N × 3) @ (3 × 5) = N × 5. Matches $A_1$)*

**Step 4: Hidden Activation Derivative (Sigmoid)**

$$\frac{\partial L}{\partial Z_1} = \frac{\partial L}{\partial A_1} \odot (A_1 \odot (1 - A_1))$$
*(Shape Check: N × 5. Note: If the exam asks for ReLU, replace the right side of the Hadamard product $\odot$ with $\mathbb{I}(Z_1 > 0)$)*

**Step 5: Hidden Weights and Biases**

$$\frac{\partial L}{\partial W_1} = X^T \frac{\partial L}{\partial Z_1}$$
*(Shape Check: (2 × N) @ (N × 5) = 2 × 5. Matches $W_1$)*

$$\frac{\partial L}{\partial b_1} = \sum_{i=1}^{N} \left(\frac{\partial L}{\partial Z_1}\right)_i$$
*(Shape Check: 1 × 5)*

---

### Part 2: Code Implementation (The Laptop Exam)

This is the modular, numerically stable implementation of the math above. Keep these cells separated in your Jupyter notebook.

**Cell 1: Numerically Stable Activations & Loss**

In [1]:
import numpy as np

def sigmoid(z):
    # Clip to prevent overflow
    z_clipped = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z_clipped))

def softmax(z):
    # Subtract max for numerical stability before exponentiation
    z_shifted = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def compute_cross_entropy(P, T):
    N = P.shape[0]
    # Add epsilon to prevent log(0)
    epsilon = 1e-15
    P_clipped = np.clip(P, epsilon, 1 - epsilon)
    return -np.sum(T * np.log(P_clipped)) / N

**Cell 2: Forward & Backward Functions**

In [2]:
def forward_pass(X, W1, b1, W2, b2):
    Z1 = np.dot(X, W1) + b1
    A1 = sigmoid(Z1)
    Z2 = np.dot(A1, W2) + b2
    P = softmax(Z2) # P replaces Y from the MSE version
    return Z1, A1, Z2, P

def backward_pass(X, P, T, A1, W2):
    N = X.shape[0]
    
    # 1. Gradient of Loss w.r.t Logits (Z2)
    dZ2 = (1 / N) * (P - T)
    
    # 2. Gradients for Output Layer
    dW2 = np.dot(A1.T, dZ2)
    db2 = np.sum(dZ2, axis=0, keepdims=True)
    
    # 3. Propagate Error to Hidden Layer
    dA1 = np.dot(dZ2, W2.T)
    
    # 4. Gradient of Hidden Activation (Sigmoid)
    dZ1 = dA1 * (A1 * (1 - A1)) 
    # NOTE: If the exam specifies ReLU, change the line above to:
    # dZ1 = dA1 * (A1 > 0)
    
    # 5. Gradients for Hidden Layer
    dW1 = np.dot(X.T, dZ1)
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    
    return dW1, db1, dW2, db2

**Cell 3: The Training Loop**

In [3]:
# Assuming X_train, T_train, W1, b1, W2, b2 are already initialized
lr = 0.50 # Cross-entropy often supports a higher learning rate than MSE
epochs = 1500

for epoch in range(epochs):
    # Forward
    Z1, A1, Z2, P = forward_pass(X_train, W1, b1, W2, b2)
    
    # Loss
    loss = compute_cross_entropy(P, T_train)
    
    # Backward
    dW1, db1, dW2, db2 = backward_pass(X_train, P, T_train, A1, W2)
    
    # Update Weights
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Cross-Entropy Loss: {loss:.5f}")

NameError: name 'X_train' is not defined

### Critical Checks During the Exam

1. **Check your matrix shapes immediately:** If `np.dot()` throws an error, do not guess transpositions. Write the shapes on paper: [(2xN) @ (Nx5)](cci:1://file:///tmp/create_nb_q1_prep.py:23:0-28:6). If the inner dimensions do not match, your chain rule derivation is out of order.
2. **Watch the Loss:** If the Cross-Entropy loss starts at `NaN` or immediately explodes, it means your Softmax function is encountering extreme values. Ensure the `z - np.max(z)` stability shift is implemented.
3. **Read the Activation Function Requirement:** Do not blindly copy Sigmoid if the prompt asks for ReLU or Tanh. You must modify both the forward `A1 = ...` and the backward local derivative `dZ1 = ...` accordingly.